In [1]:
import numpy as np
import pandas
import bgshr
import matplotlib.pyplot as plt

In [2]:
def apply_threshold(df, thresh):
    # mask windows with num_sites < thresh
    mask = np.where(df["num_sites"] < thresh)[0]
    for col in df.columns[3:]:
        df.loc[mask, col] = np.nan
    return df


example = pandas.DataFrame({
    "chrom": [0]*10, "chromStart": [0]*10, "chromEnd": [0]*10,
    "num_sites": np.arange(10), "value": np.ones(10)})
print(apply_threshold(example, 5).to_string(index=False))

 chrom  chromStart  chromEnd  num_sites  value
     0           0         0        NaN    NaN
     0           0         0        NaN    NaN
     0           0         0        NaN    NaN
     0           0         0        NaN    NaN
     0           0         0        NaN    NaN
     0           0         0        5.0    1.0
     0           0         0        6.0    1.0
     0           0         0        7.0    1.0
     0           0         0        8.0    1.0
     0           0         0        9.0    1.0


In [4]:
# Standardized plots
scale = 1e5
thresh = 50000
chrom = 22
xtick_spacing = 5e7
mut_models = ["roulette", "gnomad", "carlson"]

cons_models = [
    "merged_cds_regulatory",
    "split_cds_regulatory",
    "merged_cds_phastcons",
    "split_cds_phastcons"]

for chrom in range(15, 23):
    for mut_model in mut_models:
        fnames = [f"../models/tables_10kb/equilibrium_granular_Ne/{cons_model}/{mut_model}/B_tbl_YRI_chr{chrom}_10kb.csv.gz"
                  for cons_model in cons_models]
        dfs = [pandas.read_csv(x) for x in fnames]
        dfs = [bgshr.Util.scale_genome_table(df, scale) for df in dfs]
        dfs = [apply_threshold(df, thresh) for df in dfs]
        
        xs = np.mean([dfs[0]["chromStart"], dfs[0]["chromEnd"]], 0)
        xticks = np.arange(xs[0], xs[-1], int(xtick_spacing))
        
        fig, axs = plt.subplots(2, 2, figsize=(15, 8), layout="constrained", height_ratios=[2,1], sharey="row", sharex=True)
        kwargs = {"lw": 0.8, "marker": "o", "ms": 1}
        
        axs[0, 0].set_ylabel("$\pi$")
        axs[0, 0].plot(xs, dfs[0]["avg_pi"], label="observed", color="black", **kwargs)
        axs[0, 0].plot(xs, dfs[0]["exp_pi"], label=cons_models[0], **kwargs)
        axs[0, 0].plot(xs, dfs[1]["exp_pi"], label=cons_models[1], **kwargs)
        axs[0, 0].legend(ncols=3, fontsize="x-small")
            
        axs[0, 1].plot(xs, dfs[2]["avg_pi"], label="observed", color="black", **kwargs)
        axs[0, 1].plot(xs, dfs[2]["exp_pi"], label=cons_models[2], **kwargs)
        axs[0, 1].plot(xs, dfs[3]["exp_pi"], label=cons_models[3], **kwargs)
        axs[0, 1].legend(ncols=3, fontsize="x-small")
        
        axs[1, 0].set_ylabel("$B$")
        axs[1, 0].plot(xs, dfs[0]["B"], label=cons_models[0], **kwargs)
        axs[1, 0].plot(xs, dfs[1]["B"], label=cons_models[1], **kwargs)
        axs[1, 0].legend(ncols=3, fontsize="x-small")
            
        axs[1, 1].plot(xs, dfs[2]["B"], label=cons_models[2], **kwargs)
        axs[1, 1].plot(xs, dfs[3]["B"], label=cons_models[3], **kwargs)
        axs[1, 1].legend(ncols=3, fontsize="x-small")
        
        fig.suptitle(f"{mut_model} mutation model - chromosome {chrom}")
        plt.savefig(f"../figures/landscapes/{mut_model}_4way_chr{chrom}_100kb.pdf")
        plt.close()

In [6]:
# Standardized plots
scale = 1e6
thresh = 250000
chrom = 8
xtick_spacing = 2e7
mut_model = "roulette"

cons_models = [
    "merged_cds_regulatory",
    "split_cds_regulatory",
    "merged_cds_phastcons",
    "split_cds_phastcons"]

for chrom in range(1, 23):
    fnames = [f"../models/equilibrium_granular_Ne/{cons_model}/{mut_model}/B_tbl_YRI_chr{chrom}_10kb.csv.gz"
              for cons_model in cons_models]
    dfs = [pandas.read_csv(x) for x in fnames]
    dfs = [bgshr.Util.scale_genome_table(df, scale) for df in dfs]
    dfs = [apply_threshold(df, thresh) for df in dfs]
    
    xs = np.mean([dfs[0]["chromStart"], dfs[0]["chromEnd"]], 0)
    xticks = np.arange(xs[0], xs[-1], int(xtick_spacing))
    
    fig, axs = plt.subplots(2, 2, figsize=(15, 8), layout="constrained", height_ratios=[2,1], sharey="row", sharex=True)
    kwargs = {"lw": 0.8, "marker": "o", "ms": 1}
    
    axs[0, 0].set_ylabel("$\pi$")
    axs[0, 0].plot(xs, dfs[0]["avg_pi"], label="observed", color="black", **kwargs)
    axs[0, 0].plot(xs, dfs[0]["exp_pi"], label=cons_models[0], **kwargs)
    axs[0, 0].plot(xs, dfs[1]["exp_pi"], label=cons_models[1], **kwargs)
    axs[0, 0].legend(ncols=3, fontsize="x-small")
        
    axs[0, 1].plot(xs, dfs[2]["avg_pi"], label="observed", color="black", **kwargs)
    axs[0, 1].plot(xs, dfs[2]["exp_pi"], label=cons_models[2], **kwargs)
    axs[0, 1].plot(xs, dfs[3]["exp_pi"], label=cons_models[3], **kwargs)
    axs[0, 1].legend(ncols=3, fontsize="x-small")
    axs[0, 1].set_xticks(xticks, (xticks/1e6).astype(np.int64))
    axs[0, 1].set_xlabel("position (Mb)")
    
    axs[1, 0].set_ylabel("$B$")
    axs[1, 0].plot(xs, dfs[0]["B"], label=cons_models[0], **kwargs)
    axs[1, 0].plot(xs, dfs[1]["B"], label=cons_models[1], **kwargs)
    axs[1, 0].legend(ncols=3, fontsize="x-small")
        
    axs[1, 1].plot(xs, dfs[2]["B"], label=cons_models[2], **kwargs)
    axs[1, 1].plot(xs, dfs[3]["B"], label=cons_models[3], **kwargs)
    axs[1, 1].legend(ncols=3, fontsize="x-small")
    axs[1, 1].set_xticks(xticks, (xticks/1e6).astype(np.int64))
    axs[1, 1].set_xlabel("position (Mb)")
    
    fig.suptitle(f"{mut_model} mutation model - chromosome {chrom}")
    plt.savefig(f"../figures/{mut_model}_4way_chr{chrom}_1Mb.pdf")
    plt.close()

In [51]:
# exotic models

# Standardized plots
scale = 1e5
thresh = 50000
chrom = 22
xtick_spacing = 5e6
mut_model = "roulette"

cons_models = [
    "merged_cds_phastcons",
    "merged_cds_phastcons_75",
    "merged_cds_phastcons_80"]

fnames = [f"../models/equilibrium_granular_Ne/{cons_model}/{mut_model}/B_tbl_YRI_chr{chrom}_10kb.csv.gz"
          for cons_model in cons_models]
dfs = [pandas.read_csv(x) for x in fnames]
dfs = [bgshr.Util.scale_genome_table(df, scale) for df in dfs]
dfs = [apply_threshold(df, thresh) for df in dfs]

xs = np.mean([dfs[0]["chromStart"], dfs[0]["chromEnd"]], 0)
xticks = np.arange(xs[0], xs[-1], int(xtick_spacing))

fig, axs = plt.subplots(2, 1, figsize=(15, 8), layout="constrained", height_ratios=[2,1], sharey="row", sharex=True)
kwargs = {"lw": 0.8, "marker": "o", "ms": 1}

axs[0].set_ylabel("$\pi$")
axs[0].plot(xs, dfs[0]["avg_pi"], label="observed", color="black", **kwargs)
axs[0].plot(xs, dfs[0]["exp_pi"], label=cons_models[0], **kwargs)
axs[0].plot(xs, dfs[1]["exp_pi"], label=cons_models[1], **kwargs)
axs[0].plot(xs, dfs[2]["exp_pi"], label=cons_models[2], **kwargs)
axs[0].legend(ncols=4, fontsize="x-small")
    
axs[1].set_ylabel("$B$")
axs[1].plot(xs, dfs[0]["B"], label=cons_models[0], **kwargs)
axs[1].plot(xs, dfs[1]["B"], label=cons_models[1], **kwargs)
axs[1].plot(xs, dfs[2]["B"], label=cons_models[2], **kwargs)
axs[1].legend(ncols=3, fontsize="x-small")
axs[1].set_xticks(xticks, (xticks/1e6).astype(np.int64))
axs[1].set_xlabel("position (Mb)")

fig.suptitle(f"{mut_model} mutation model - chromosome {chrom}")
plt.savefig(f"../figures/landscapes/{mut_model}_extended_phastcons_chr{chrom}_100kb.pdf")
plt.close()

In [54]:
# exotic models

# Standardized plots
scale = 1e6
thresh = 250000
chrom = 16
xtick_spacing = 2e7
mut_model = "roulette"

cons_models = [
    "merged_cds_phastcons",
    "merged_cds_phastcons_75",
    "merged_cds_phastcons_80"]

fnames = [f"../models/equilibrium_granular_Ne/{cons_model}/{mut_model}/B_tbl_YRI_chr{chrom}_10kb.csv.gz"
          for cons_model in cons_models]
dfs = [pandas.read_csv(x) for x in fnames]
dfs = [bgshr.Util.scale_genome_table(df, scale) for df in dfs]
dfs = [apply_threshold(df, thresh) for df in dfs]

xs = np.mean([dfs[0]["chromStart"], dfs[0]["chromEnd"]], 0)
xticks = np.arange(xs[0], xs[-1], int(xtick_spacing))

fig, axs = plt.subplots(2, 1, figsize=(15, 8), layout="constrained", height_ratios=[2,1], sharey="row", sharex=True)
kwargs = {"lw": 0.8, "marker": "o", "ms": 1}

axs[0].set_ylabel("$\pi$")
axs[0].plot(xs, dfs[0]["avg_pi"], label="observed", color="black", **kwargs)
axs[0].plot(xs, dfs[0]["exp_pi"], label=cons_models[0], **kwargs)
axs[0].plot(xs, dfs[1]["exp_pi"], label=cons_models[1], **kwargs)
axs[0].plot(xs, dfs[2]["exp_pi"], label=cons_models[2], **kwargs)
axs[0].legend(ncols=4, fontsize="x-small")
    
axs[1].set_ylabel("$B$")
axs[1].plot(xs, dfs[0]["B"], label=cons_models[0], **kwargs)
axs[1].plot(xs, dfs[1]["B"], label=cons_models[1], **kwargs)
axs[1].plot(xs, dfs[2]["B"], label=cons_models[2], **kwargs)
axs[1].legend(ncols=3, fontsize="x-small")
axs[1].set_xticks(xticks, (xticks/1e6).astype(np.int64))
axs[1].set_xlabel("position (Mb)")

fig.suptitle(f"{mut_model} mutation model - chromosome {chrom}")
plt.savefig(f"../figures/landscapes/{mut_model}_extended_phastcons_chr{chrom}_1Mb.pdf")
plt.close()

In [7]:
# plot the HLA
hla_start = 28_500_000
hla_end = 33_500_000
buffer = 2_000_000

start = hla_start - buffer
end = hla_end + buffer
scale = 10_000
thresh = 1
xtick_spacing = 1_000_000
chrom = 6

mut_models = ["carlson", "gnomad", "roulette"]
mut_labels = ["Carlson", "Gnomad", "Roulette"]
cons_models = [
    "merged_cds_regulatory",
    "split_cds_regulatory",
    "merged_cds_phastcons",
    "split_cds_phastcons"]
cons_labels = [
    "merged CDS + regulatory",
    "split CDS + regulatory",
    "merged CDS + phastCons",
    "split CDS + phastCons"]

fig, axs = plt.subplots(3, 1, figsize=(10, 12), layout="constrained", sharey="row", sharex=True)

for i, mut_model in enumerate(mut_models):
    fnames = [f"../local_models/equilibrium_granular_Ne/{cons_model}/{mut_model}/{cons_model}_{mut_model}_chr{chrom}.csv.gz"
              for cons_model in cons_models]
    dfs = [pandas.read_csv(x) for x in fnames]
    pi_fname = f"../data/pi_tables/{mut_model}/pi_tbl_YRI_chr1.csv.gz"
    dfs.append(pandas.read_csv(pi_fname))
    dfs = [df[(df["chromStart"] >= start) & (df["chromEnd"] <= end)] for df in dfs]
    dfs = [bgshr.Util.scale_genome_table(df, scale) for df in dfs]
    dfs = [apply_threshold(df, thresh) for df in dfs]
    
    xs = np.mean([dfs[0]["chromStart"], dfs[0]["chromEnd"]], 0)
    xticks = np.arange(start, end + 1, int(xtick_spacing))
    
    kwargs = {"lw": 0.8, "marker": "o", "ms": 1}
    
    axs[i].set_ylabel("$\pi$")
    axs[i].plot(xs, dfs[-1]["avg_pi"], label="observed", color="black", **kwargs)
    for j in range(4):
        axs[i].plot(xs, dfs[j]["exp_pi"], label=cons_labels[j], **kwargs)
    
    axs[i].legend(ncols=1, fontsize="x-small", framealpha=0, loc="best")
    axs[i].set_xticks(xticks, (xticks/1e6))
    axs[i].set_xlabel("position (Mb)")

    axs[i].axvline(hla_start, color="black", linestyle="--", lw=0.8)
    axs[i].axvline(hla_end, color="black", linestyle="--", lw=0.8)
    axs[i].set_ylim(0, 0.0045)
    axs[i].set_xlim(start - 1e4, end + 1e4)
    axs[i].set_title(mut_labels[i])

plt.savefig(f"../figures/unmasked_HLA_region_10kb.pdf")
plt.close()

In [15]:
# 12-way plots; 100kb
scale = 100_000
thresh = 50_000
xtick_spacing = 5_000_000
chrom = 21

mut_models = ["carlson", "gnomad", "roulette"]
mut_labels = ["Carlson", "Gnomad", "Roulette"]
cons_models = [
    "merged_cds_regulatory",
    "split_cds_regulatory",
    "merged_cds_phastcons",
    "split_cds_phastcons"]
cons_labels = [
    "merged CDS + regulatory",
    "split CDS + regulatory",
    "merged CDS + phastCons",
    "split CDS + phastCons"]

fig, axs = plt.subplots(3, 1, figsize=(12, 12), layout="constrained", sharey="row", sharex=True)


for i, mut_model in enumerate(mut_models):
    fnames = [f"../models/equilibrium_granular_Ne/{cons_model}/{mut_model}/B_tbl_YRI_chr{chrom}_10kb.csv.gz"
          for cons_model in cons_models]
    dfs = [pandas.read_csv(x) for x in fnames]
    dfs = [bgshr.Util.scale_genome_table(df, scale) for df in dfs]
    dfs = [apply_threshold(df, thresh) for df in dfs]
    first_window = np.where(dfs[0]["num_sites"] > 0)[0][0]
    dfs = [df.loc[first_window:] for df in dfs]
    
    xs = np.mean([dfs[0]["chromStart"], dfs[0]["chromEnd"]], 0)
    xticks = np.arange(int(xs[0] / 1e6) * 1e6, xs[-1] + 1, int(xtick_spacing))
    kwargs = {"lw": 0.8, "marker": "o", "ms": 1}
    
    axs[i].set_ylabel("$\pi$")
    axs[i].plot(xs, dfs[-1]["avg_pi"], label="observed", color="black", **kwargs)
    for j in range(4):
        axs[i].plot(xs, dfs[j]["exp_pi"], label=cons_labels[j], **kwargs)
    
    axs[i].legend(ncols=1, fontsize="x-small", framealpha=0, loc="best")
    axs[i].set_xticks(xticks, (xticks/1e6))
    axs[i].set_xlabel("position (Mb)")

    axs[i].set_xlim(xs[0] - 1e6, xs[-1] + 1e6)
    axs[i].set_title(mut_labels[i])

plt.savefig(f"../figures/landscapes/12way_chr{chrom}_100kb.pdf")
plt.close()

In [6]:
# 12-way plots; 1Mb
scale = 1_000_000
thresh = 250_000
xtick_spacing = 20_000_000

mut_models = ["carlson", "gnomad", "roulette"]
mut_labels = ["Carlson", "Gnomad", "Roulette"]
cons_models = [
    "merged_cds_regulatory",
    "split_cds_regulatory",
    "merged_cds_phastcons",
    "split_cds_phastcons"]
cons_labels = [
    "merged CDS + regulatory",
    "split CDS + regulatory",
    "merged CDS + phastCons",
    "split CDS + phastCons"]

for chrom in range(1, 23):
    fig, axs = plt.subplots(3, 1, figsize=(12, 12), layout="constrained", sharey="row", sharex=True)
    
    
    for i, mut_model in enumerate(mut_models):
        fnames = [f"../models/tables_10kb/equilibrium_granular_Ne/{cons_model}/{mut_model}/B_tbl_YRI_chr{chrom}_10kb.csv.gz"
              for cons_model in cons_models]
        dfs = [pandas.read_csv(x) for x in fnames]
        dfs = [bgshr.Util.scale_genome_table(df, scale) for df in dfs]
        dfs = [apply_threshold(df, thresh) for df in dfs]
        first_window = np.where(dfs[0]["num_sites"] > 0)[0][0]
        dfs = [df.loc[first_window:] for df in dfs]
        
        xs = np.mean([dfs[0]["chromStart"], dfs[0]["chromEnd"]], 0)
        xticks = np.arange(int(xs[0] / 1e6) * 1e6, xs[-1] + 1, int(xtick_spacing))
        kwargs = {"lw": 0.8, "marker": "o", "ms": 1}
        
        axs[i].set_ylabel("$\pi$")
        axs[i].plot(xs, dfs[-1]["avg_pi"], label="observed", color="black", **kwargs)
        for j in range(4):
            axs[i].plot(xs, dfs[j]["exp_pi"], label=cons_labels[j], **kwargs)
        
        axs[i].legend(ncols=1, fontsize="x-small", framealpha=0, loc="best")
        axs[i].set_xticks(xticks, (xticks/1e6))
        axs[i].set_xlabel("position (Mb)")
    
        axs[i].set_xlim(xs[0] - 1e6, xs[-1] + 1e6)
        axs[i].set_title(mut_labels[i])
    
    plt.savefig(f"../figures/landscapes/12way_chr{chrom}_1Mb.pdf")
    plt.close()